# Stacked Survival Analysis of Rotterdam & GBSG data

This analysis follows the analysis done in [A review of survival stacking: a method to
cast survival regression analysis as classification problem][sspaper] by Erin Craig *et al.*

[sspaper]: https://doi.org/10.1515/ijb-2022-0055

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

In [2]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

In [3]:
from sksurv.metrics import integrated_brier_score

## Load data

### Rotterdam

From [Flexible Parametric Survival Analysis Using Stata](https://www.stata-press.com/data/fpsaus.html)

In [4]:
def load_rotterdamn():
    import fsspec
    url = 'https://www.stata-press.com/data/fpsaus/fpsaus.zip'
    member = 'fpsaus/rott2.dta'
    with fsspec.open(f"zip://{member}::{url}", mode="rb") as fh:
        df = pd.read_stata(fh)
    df = df[df.nodes > 0]
    df['event.time'] = np.nan
    df.loc[df['osi'] == "deceased", 'event.time'] = df.loc[df['osi'] == "deceased", 'os']
    df.loc[df['mfi'] == "yes", 'event.time'] = df.loc[df['mfi'] == "yes", 'mf']
    df.loc[df['rfi'] == 1, 'event.time'] = df.loc[df['rfi'] == 1, 'rf']
    df['event'] = (
        (df['event.time'] < 84) &
        ((df['rfi'] == 1) | (df['osi'] == "deceased") | (df['mfi'] == "yes"))
    )
    mask = df['event'] == 0
    df.loc[mask, 'event.time'] = df.loc[mask, ['rf', 'os', 'mf']].min(axis=1).clip(upper=84)
    
    df['hormon'] = df['hormon'].map({'yes': 1, 'no': 0})
    df['meno'] = df['meno'].map({'post': 1, 'pre': 0})
    df['event.time.rounded'] = (df['event.time'].round(1) * 100).astype(int)
    return df

In [5]:
rd = load_rotterdamn()
rd.shape

(1546, 28)

### German Breast Cancer Study Group (GBSG)

Imported from the R package *survival*

In [6]:
def load_gbsg():
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message='.*R_SESSION_TMPDIR.*')
        from rpy2.robjects import r, pandas2ri
        from rpy2.robjects.packages import importr
        from rpy2.robjects.conversion import localconverter
    survival = importr('survival')
    r('data(package="survival")')
    with localconverter(pandas2ri.converter):
        df = r['gbsg']
    
    df = df[df.nodes > 0]
    df['event.time'] = df.rfstime/30.437
    df = df.rename(columns={"pgr": "pr", "status": "event", "pid": "id"})
    df['event.time.rounded'] = (df['event.time'].round(1) * 100).astype(int)
    return df
    

In [7]:
gb = load_gbsg()
gb.shape

(686, 13)

### Create training and test sets

In [8]:
X_cols = ["age", "grade", "nodes", "pr", "er", "meno", "hormon"]

In [9]:
X_train = rd[X_cols].values
y_train = np.array(list(zip(rd['event'], rd['event.time.rounded'])), 
                   dtype=[('cens', '?'), ('time', '<f8')])

In [10]:
X_test = gb[X_cols].values
y_test = np.array(list(zip(gb['event'], gb['event.time.rounded'])), 
                   dtype=[('cens', '?'), ('time', '<f8')])

## Survival Analyses

In [11]:
results = []

### scikit-survival

In [12]:
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest

def train_and_evaluate_sksurv(clf, X_train, y_train, X_test, y_test):
    _ = clf.fit(X_train, y_train)
    times = y_train[y_train['cens'] == 1]['time']
    test_survival = clf.predict_survival_function(X_test, return_array=True)
    times = test_survival
    return integrated_brier_score(
        survival_train=y_train,
        survival_test=y_test,
        estimate=test_survival,
        times=clf.unique_times_
    )

#### Cox Regression

In [13]:
%%time
clf = CoxPHSurvivalAnalysis(
    n_iter=1000,
    alpha=1.0,
)

results.append({
    'encoding': 'sksurv',
    'classifier': 'CoxPHSurvivalAnalysis',
    'IBS': train_and_evaluate_sksurv(clf, X_train, y_train, X_test, y_test)
})

CPU times: user 164 ms, sys: 10.3 ms, total: 174 ms
Wall time: 171 ms


#### Survival Random Forest

In [14]:
%%time
clf = RandomSurvivalForest(
    n_estimators=100,
    max_depth=3,
    max_features=5,
    n_jobs=-1
)
results.append({
    'encoding': 'sksurv',
    'classifier': 'RandomSurvivalForest',
    'IBS': train_and_evaluate_sksurv(clf, X_train, y_train, X_test, y_test)
})

CPU times: user 2.07 s, sys: 261 ms, total: 2.33 s
Wall time: 433 ms


### Survival Stacker

In [15]:
from survstack import SurvivalStacker

In [16]:
def train_and_evaluate(stacker, clf, X_train, y_train, X_test, y_test):
    X_train_stack, y_train_stack = stacker.fit_transform(X_train, y_train)
    X_test_stack, y_test_stack = stacker.transform(X_test)
    _ = clf.fit(X_train_stack, y_train_stack)
    test_estimates = clf.predict_proba(X_test_stack)[:,1]
    test_survival = stacker.predict_survival_function(test_estimates)
    return integrated_brier_score(
        survival_train=y_train,
        survival_test=y_test,
        estimate=test_survival,
        times=stacker.times
    )

### Stacking: One-Hot Encoding

The `one-hot` `time_encoding` option represents each discrete time point as a separate binary feature, as in the original proposed algorithm. This approach makes no assumptions about the functional form of risk over time and allows the model to learn hazards independently at each time point.

In [17]:
ss_ohot = SurvivalStacker(time_encoding='onehot')

#### Random Forest

In [18]:
%%time
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    max_features=5,
    n_jobs=-1
)
results.append({
    'encoding': 'onehot',
    'classifier': 'RandomForestClassifier',
    'IBS': train_and_evaluate(ss_ohot, clf, X_train, y_train, X_test, y_test)
})

CPU times: user 22.9 s, sys: 6.57 s, total: 29.5 s
Wall time: 8.88 s


#### Gradient Boosting

In [19]:
%%time
clf = HistGradientBoostingClassifier(
    max_iter=1000,
    learning_rate=0.1,  
    max_depth=3, 
    early_stopping=True
)
results.append({
    'encoding': 'onehot',
    'classifier': 'GradientBoostingClassifier',
    'IBS': train_and_evaluate(ss_ohot, clf, X_train, y_train, X_test, y_test)
})

CPU times: user 1min 33s, sys: 3.04 s, total: 1min 36s
Wall time: 10.3 s


#### Neural Network

In [20]:
%%time
clf = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='tanh',
    solver='adam',
    batch_size=2000,
    max_iter=50,
    early_stopping=True,
)
results.append({
    'encoding': 'onehot',
    'classifier': 'MLPClassifier',
    'IBS': train_and_evaluate(ss_ohot, clf, X_train, y_train, X_test, y_test)
})

CPU times: user 5min 52s, sys: 5.12 s, total: 5min 57s
Wall time: 24.8 s


### Stacking: Continuous Encoding

The `continuous` `time_encoding` option represents time as a single numeric feature. This is far more compact and computationally efficient when there are many distinct event times, but it implicitly imposes a functional form on how the baseline hazard varies with time. Use it when you need scalability and are willing to trade some nonparametric flexibility for far fewer features.

In [21]:
ss_cont = SurvivalStacker(time_encoding='continuous')

#### Random Forest

In [22]:
%%time
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    max_features=5,
    n_jobs=-1
)
results.append({
    'encoding': 'continous',
    'classifier': 'RandomForestClassifier',
    'IBS': train_and_evaluate(ss_cont, clf, X_train, y_train, X_test, y_test)
})

CPU times: user 58.4 s, sys: 1.12 s, total: 59.5 s
Wall time: 4.11 s


#### Gradient Boosting

In [23]:
%%time
clf = HistGradientBoostingClassifier(
    max_iter=1000,
    learning_rate=0.1,  
    max_depth=3, 
    early_stopping=True
)
results.append({
    'encoding': 'continous',
    'classifier': 'GradientBoostingClassifier',
    'IBS': train_and_evaluate(ss_cont, clf, X_train, y_train, X_test, y_test)
})

CPU times: user 9.63 s, sys: 43.3 ms, total: 9.67 s
Wall time: 925 ms


#### Neural Network

In [24]:
%%time
clf = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='tanh',
    solver='adam',
    batch_size=2000,
    max_iter=50,
    early_stopping=True,
)
results.append({
    'encoding': 'continous',
    'classifier': 'MLPClassifier',
    'IBS': train_and_evaluate(ss_cont, clf, X_train, y_train, X_test, y_test)
})

CPU times: user 1min 2s, sys: 78.3 ms, total: 1min 2s
Wall time: 8.19 s


## Results

The results indicate that the survival-specific models, particularly Random Survival Forest, achieve the strongest overall performance, with Cox proportional hazards regression performing slightly worse but still competitive. Among the classification-based models, the one-hot encoding strategy generally outperforms continuous encoding, with Gradient Boosting using one-hot features coming closest to the survival models. Random Forest classifiers also perform reasonably well under one-hot encoding, while MLP classifiers tend to lag behind, especially with continuous encoding, where performance degradation is substantial. This suggests that encoding strategy plays a critical role in adapting classification models for survival tasks, and survival-specific models still hold a notable advantage.

In [25]:
pd.DataFrame(results)

,encoding,classifier,IBS
0,sksurv,CoxPHSurvivalAnalysis,0.123951
1,sksurv,RandomSurvivalForest,0.115955
2,onehot,RandomForestClassifier,0.130044
3,onehot,GradientBoostingClassifier,0.119938
4,onehot,MLPClassifier,0.130975
5,continous,RandomForestClassifier,0.119892
6,continous,GradientBoostingClassifier,0.168062
7,continous,MLPClassifier,0.381264
